# 06.COMPLETO · Fase 2: efectos entre categorías (6 categorías sin programa semanal)

Estima, para cada categoría objetivo $j$, el modelo de la fase 1 más la promoción de las otras categorías del núcleo y la variable de control `p__RESTO` (ecuación de la "Fase 2" en `metodologia.tex`; D‑07, D‑15, D‑17, D‑28). Responde la pregunta (iii): ¿la promoción de otras categorías aumenta (sinergia) o disminuye (canibalización) las ventas de los SKUs de la categoría $j$?

| | |
|---|---|
| **Entrada** | `panel_t44_sku_dia.parquet`, `exposiciones_t44_sku_dia.parquet`, `calendario_t44.csv`, `exposicion_t44_categorias_dia.csv`, `exposicion_t44_resto_por_objetivo.csv` (de `01` a `03`) y `tabla_07_fase1_consolidada.csv` (de `05`, solo para comparar) |
| **Salidas** | `tabla_11_fase2_coeficientes.csv`, `tabla_12_fase2_efectos_cruzados.csv`, `tabla_13_fase2_bases_y_controles.csv` en `resultados/tablas/` |
| **Decisiones que implementa** | D‑07, D‑08, D‑09, D‑14, D‑15, D‑16, D‑17, D‑26, D‑28 |

**Qué se agrega a la fase 1.**

1. De `exposicion_t44_categorias_dia.csv`, **solo** las columnas de las 10 categorías del núcleo (`p__<CATEGORÍA>`, `_l1_3`, `_l4_6`), unidas por `date`.
2. Se **elimina la columna de la propia categoría $j$**: su promoción ya está en las variables propias y de clase.
3. De `exposicion_t44_resto_por_objetivo.csv`, `p__RESTO` y sus bloques para la categoría $j$ (unidos por `date` y `family`), como **control**. Su coeficiente no se interpreta.

**Cómo se lee el resultado.** Para cada par $(j, k)$, $L_{jk}=e^{0{,}1\,\beta_{jk}}-1$ es el cambio de las ventas de $j$ cuando 10 puntos porcentuales adicionales del surtido de $k$ están en promoción el mismo día. Se aplica la corrección de Benjamini‑Hochberg (FDR 5 %) sobre **todos** los efectos cruzados contemporáneos (54 con 6 categorías objetivo).

**Errores estándar (importante).** Con agrupamiento en dos vías, la matriz de varianzas de `pyfixest` puede no ser semidefinida positiva. Los resultados principales usan la corrección de Cameron, Gelbach y Miller (2011), que lleva a cero los valores propios negativos; las tablas guardan también el error estándar y el valor p **sin corregir** (`*_sin_corregir`) y la clasificación que resultaría con ellos. Si ambas versiones difieren en qué pares son significativos, es una decisión que debe tomar el profesor guía.

En las categorías con programa semanal (MEATS, POULTRY, DELI, PRODUCE) el $P_{kt}$ mide solo las promociones **fuera del programa**. Esas categorías no son objetivo en este notebook.

**Versión COMPLETA (qué junta).** Combina lo mejor de dos versiones:
- **Del notebook de la compañera (06.5):** solo las **6 categorías objetivo sin programa semanal** (BEVERAGES, GROCERY I, CLEANING, DAIRY, BREAD/BAKERY y LIQUOR,WINE,BEER), el diagnóstico de nulos por categoría, la igualdad exacta de días usados (532) y los controles de errores estándar.
- **Del notebook 06 explicado:** la corrección de la matriz de varianzas (con la versión **sin corregir** guardada como sensibilidad), el efecto neto (hoy + rezagos), la verificación con otro programa (statsmodels), los controles de estabilidad frente a la fase 1 y las explicaciones para principiantes.

**Por qué solo 6 categorías.** En las 4 categorías con programa semanal (MEATS, POULTRY, DELI, PRODUCE) los rezagos están contaminados por el programa, y esos resultados no se interpretan hasta que el profesor guía decida la regla (D‑14). **Sí se usan como categorías en promoción** (columnas de la matriz), midiendo solo sus promociones fuera del programa. Así hay 6 x 9 = **54 pares** (no 90).

**Salidas nuevas:** `tabla_28_fase2c_coeficientes.csv`, `tabla_29_fase2c_efectos_cruzados.csv`, `tabla_30_fase2c_bases_y_controles.csv`, `tabla_31_fase2c_comparacion_con_fase2_de_10.csv`, `tabla_32_fase2c_sensibilidad_solo_semana.csv` y `tabla_33_fase2c_nulos_estructurales_clase.csv`. No se modifica ninguna tabla anterior.


## Guía para leer este notebook (si es la primera vez que ves algo así)

**La pregunta.** En el notebook 05 vimos qué pasa con las ventas de un producto cuando **ese producto** (o sus parecidos) está en promoción. Aquí hacemos la pregunta de al lado: **¿qué pasa con las ventas de las bebidas cuando la cadena hace promoción en pan, en limpieza o en carnes?**
- Si las ventas de bebidas **suben**, hay **sinergia** (la promoción de otra categoría "arrastra" clientes hacia bebidas).
- Si las ventas de bebidas **bajan**, hay **canibalización** (la promoción de otra categoría se lleva clientes que habrían comprado bebidas).
- Si **no cambian**, no hay efecto.

**La idea, con una analogía.** Piensa en una feria con puestos de frutas, pan y bebidas. Un día el puesto de pan pone un cartel de oferta. Nos preguntamos si el puesto de bebidas vende más (llegó más gente), menos (la gente gastó en pan) o igual. Para no confundirnos con otras cosas (feriados, fines de semana, el nivel de cada puesto), comparamos cada producto consigo mismo, igual que en el notebook 05.

**Qué se agrega al modelo del notebook 05.** El mismo modelo, más las promociones de las **otras 9 categorías principales** (el "núcleo"), cada una con su efecto de hoy y de días anteriores (rezagos), más un control llamado `p__RESTO` (la promoción en todas las demás categorías juntas).

| Palabra | Qué significa, en simple |
|---|---|
| **Categoría objetivo (j)** | La categoría cuyas ventas queremos explicar (por ejemplo, BEVERAGES). |
| **Categoría en promoción (k)** | La otra categoría cuya promoción miramos (por ejemplo, BREAD/BAKERY). |
| **Par (j, k)** | Una combinación "k afecta a j". Aquí hay 6 categorías afectadas × 9 en promoción = **54 pares**. |
| **`p__<CATEGORÍA>`** | Qué proporción (de 0 a 1) del surtido de esa categoría estaba en promoción ese día. 0,10 = 10 % del surtido en promoción. |
| **Núcleo** | Las 10 categorías principales que se usan como "otras categorías". |
| **`p__RESTO`** | Un control: promoción promedio de todo lo que no es del núcleo. No se interpreta, solo ayuda a limpiar el resultado. |
| **L_jk** | El resultado final de cada par: cuánto cambian las ventas de j cuando 10 puntos porcentuales más del surtido de k están en promoción. |
| **Valor p** | Probabilidad de ver un resultado así de grande si en realidad no hubiera efecto. Menor que 0,05 → "difícilmente casualidad". |
| **Valor q (Benjamini‑Hochberg)** | Un valor p "más exigente", que descuenta que probamos **muchos pares a la vez**. Se explica en la sección 5. |

**Por qué hace falta ser más exigente (la idea del "billete de lotería").** Si pruebas 54 pares con un criterio de 5 %, esperas que unos 2 o 3 salgan "significativos" **por pura suerte**, como quien compra 90 billetes de lotería y alguno sale premiado. La corrección de Benjamini‑Hochberg sirve para no creer en esos premios de casualidad.

In [1]:
# ------------------------------------------------------------------------------------------------------------
# Este notebook estima la fase 2 de la metodología: efectos de la promoción de otras categorías del núcleo
# sobre las ventas de cada SKU, con PPML y efectos fijos.
# ------------------------------------------------------------------------------------------------------------
# Fecha de versión: 2026-10-22
# ------------------------------------------------------------------------------------------------------------

# Librerías
# ------------------------------------------------------------------------------------------------------------
# Librerías: cada línea trae una "caja de herramientas" que usaremos más abajo.
import os                                               # carpetas y rutas de archivos
import re                                               # limpiar textos (nombres de categorías)
import warnings                                         # controlar los avisos de Python

import numpy as np                                      # cálculos matemáticos
import pandas as pd                                     # tablas de datos (como un Excel dentro de Python)
import pyfixest as pf                                   # estima el modelo estadístico (Poisson con efectos fijos)
from scipy import stats                                 # distribuciones de probabilidad (valores p e intervalos)
from statsmodels.stats.multitest import multipletests   # la corrección de Benjamini-Hochberg por comparaciones múltiples

# Aviso de pyfixest "invalid value encountered in sqrt": pyfixest saca la raíz de una varianza que a veces sale ligeramente negativa.
# No es un error: la función `inferencia` corrige esa matriz (Cameron, Gelbach y Miller, 2011) y los resultados oficiales salen de la versión corregida.
# Se silencia solo ese aviso; los demás (por ejemplo "observations removed because of separation") siguen apareciendo.
warnings.filterwarnings("ignore", message=".*invalid value encountered in sqrt.*")


In [2]:
# Parámetros
# ------------------------------------------------------------------------------------------------------------
# Categorías objetivo (las mismas de la fase 1) y núcleo de exposición (D-28)
# Categorías cuyas ventas queremos explicar: las 6 SIN programa semanal (las mismas que usa la compañera en 06.5 y que usamos en 05.COMPLETO).
categorias_objetivo = ["BEVERAGES", "GROCERY I", "CLEANING", "DAIRY", "BREAD/BAKERY", "LIQUOR,WINE,BEER"]
# El "núcleo": las 10 categorías cuya promoción usamos como explicación. Para cada objetivo se usan las otras 9.
nucleo_exposicion = [
    "MEATS", "POULTRY", "DELI", "PRODUCE", "DAIRY",
    "BREAD/BAKERY", "BEVERAGES", "LIQUOR,WINE,BEER", "GROCERY I", "CLEANING",
]
# Categorías con programa semanal fijo (casi todo en promoción el mismo día cada semana). Sus días de programa se sacan del análisis.
categorias_programa = ["MEATS", "POULTRY", "SEAFOOD", "EGGS", "DELI", "PRODUCE"]   # D-26
assert not set(categorias_objetivo) & set(categorias_programa), "no debe haber categorías con programa semanal entre las objetivo (rezagos contaminados, pendiente D-14)"

# D-14: el día actual y los bloques de rezagos
# Para cada promoción se usan 3 versiones: hoy (""), promedio de 1 a 3 días atrás ("_l1_3") y promedio de 4 a 6 días atrás ("_l4_6").
sufijos_bloques = ["", "_l1_3", "_l4_6"]
# Las tres promociones "propias" (igual que en el notebook 05): el propio producto, su clase y el resto de su categoría.
variables_base = ["onpromotion", "promo_clase_otros", "promo_cat_otras_clases"]

# Variables de calendario (feriados, días de pago, eventos) para no confundir sus efectos con los de la promoción.
variables_calendario = [
    "dia_libre", "adicional", "dia_laboral_recuperado", "evento_comercial", "evento_otro", "dia_pago", "post_pago",
]
# Cada producto, cada día de la semana y cada mes tienen su propio "nivel normal" (efectos fijos): así se compara cada producto consigo mismo.
efectos_fijos = "item_nbr + dow + mes"             # D-16
# Errores estándar agrupados por producto y por semana: reconoce que las ventas de un mismo producto o de una misma semana se parecen entre sí.
agrupamiento = "item_nbr + semana"                 # D-15
# Tope de días con datos (544). Si una base usa más, falta una exclusión.
dias_maximos = 544
dias_esperados = 532                                # 590 días con operación - 46 del terremoto - 12 sin dato por los rezagos
# Alarma: un coeficiente mayor que 5 en valor absoluto suele indicar un problema técnico.
umbral_coeficiente = 5
# Tasa de falsos descubrimientos del 5 %: se acepta que, de los pares marcados como efecto, como máximo ~5 % sean casualidad.
tasa_fdr = 0.05                                    # D-15: Benjamini-Hochberg
# Se mide el efecto de +10 puntos porcentuales (0,1) del surtido en promoción; subir de 0 % a 100 % no es realista.
cambio_pp = 0.1                                    # efecto de 10 puntos porcentuales: L = exp(0,1 beta) - 1

# Rutas
# ------------------------------------------------------------------------------------------------------------
# Rutas de los archivos: datos ya procesados en ../data/procesados; resultados en resultados/tablas.
ruta_procesados = os.path.join("..", "data", "procesados")
archivo_panel = os.path.join(ruta_procesados, "panel_t44_sku_dia.parquet")
archivo_exposiciones = os.path.join(ruta_procesados, "exposiciones_t44_sku_dia.parquet")
archivo_calendario = os.path.join(ruta_procesados, "calendario_t44.csv")
archivo_categorias = os.path.join(ruta_procesados, "exposicion_t44_categorias_dia.csv")
archivo_resto = os.path.join(ruta_procesados, "exposicion_t44_resto_por_objetivo.csv")
ruta_tablas = os.path.join("resultados", "tablas")
archivo_fase1 = os.path.join(ruta_tablas, "tabla_07_fase1_consolidada.csv")
os.makedirs(ruta_tablas, exist_ok=True)

## 1. Funciones auxiliares

In [3]:
def nombre_archivo(categoria: str) -> str:
    """
    Esta función convierte el nombre de una categoría en un texto apto para nombres de archivo.

    :param categoria:   el nombre de la categoría, por ejemplo "LIQUOR,WINE,BEER".
    :return:            el nombre sin símbolos, por ejemplo "LIQUOR_WINE_BEER".
    """
    # Reemplaza espacios, comas y símbolos por guiones bajos: "LIQUOR,WINE,BEER" -> "LIQUOR_WINE_BEER".
    return re.sub(r"[^A-Z0-9]+", "_", categoria.upper()).strip("_")


def columna_exposicion(categoria: str) -> str:
    """
    Esta función devuelve el nombre de la columna de intensidad promocional de una categoría, tal como la crea
    el notebook 03 (por ejemplo, "LIQUOR,WINE,BEER" -> "p__LIQUOR_WINE_BEER").

    :param categoria:   el nombre de la categoría.
    :return:            el nombre de la columna p__<CATEGORIA>.
    """
    # Agrega el prefijo "p__": así se llama la columna con la proporción del surtido en promoción.
    return "p__" + nombre_archivo(categoria)


def a_porcentaje(beta, factor: float = 1.0):
    """
    Esta función convierte un coeficiente en el cambio porcentual de las ventas, 100 (exp(factor * beta) - 1).

    :param beta:    un coeficiente o una serie de coeficientes.
    :param factor:  el cambio de la variable al que se refiere el efecto (1 = una unidad, 0.1 = 10 puntos porcentuales).
    :return:        el cambio porcentual de las ventas.
    """
    # La fórmula clave: un coeficiente beta se traduce a % con 100 x (e^(factor x beta) - 1).
    # Ejemplo: beta = 0,47 -> e^0,47 = 1,60 -> las ventas son 1,6 veces las normales -> +60 %.
    # Con factor = 0,1 se obtiene el efecto de +10 puntos porcentuales.
    return 100 * (np.exp(factor * np.asarray(beta, dtype=float)) - 1)


def inferencia(modelo) -> dict:
    """
    Esta función toma la matriz de varianzas con agrupamiento en dos vías y, si no es semidefinida positiva,
    la corrige llevando a cero sus valores propios negativos (Cameron, Gelbach y Miller, 2011). Con agrupamiento
    en dos vías la matriz puede salir con varianzas negativas, y pyfixest entrega entonces un error estándar nulo (NaN).
    Si la matriz ya es semidefinida positiva, no cambia.

    :param modelo:  el modelo ajustado de pyfixest.
    :return:        un diccionario con los nombres, los coeficientes, la matriz de varianzas, los grados de libertad
                    y un indicador de si hubo que corregir la matriz.
    """
    # La "matriz de varianzas" resume cuánta incertidumbre tiene cada coeficiente y cómo se relacionan entre sí.
    vcov = np.asarray(modelo._vcov, dtype=float)
    vcov = (vcov + vcov.T) / 2
    # Se revisa si la matriz es válida: sus "valores propios" no pueden ser negativos (sería una varianza negativa, que no existe).
    valores, vectores = np.linalg.eigh(vcov)
    # True si hubo valores propios negativos (matriz inválida).
    corregida = bool(valores.min() < -1e-12 * max(abs(valores).max(), 1e-300))
    if corregida:
        # Corrección de Cameron, Gelbach y Miller (2011): los valores negativos se llevan a cero. Si la matriz era válida, no cambia nada.
        vcov = (vectores * np.clip(valores, 0, None)) @ vectores.T
    # Grados de libertad: definen qué tan "ancha" es la curva t usada para valores p e intervalos (dependen del número de grupos).
    gl = getattr(modelo, "_df_t", None)
    gl = float(gl) if gl is not None and np.isfinite(gl) else np.inf
    return {
        "nombres": [str(c) for c in modelo._coefnames], "beta": np.asarray(modelo._beta_hat, dtype=float).ravel(),
        "vcov": vcov, "gl": gl, "corregida": corregida,
    }


def resumen_coeficiente(beta: float, varianza: float, gl: float) -> dict:
    """
    Esta función calcula error estándar, valor p e intervalo de confianza del 95 % de un coeficiente (o de una suma).

    :param beta:        el coeficiente.
    :param varianza:    su varianza.
    :param gl:          los grados de libertad de la distribución t (infinito = normal).
    :return:            un diccionario con coef, se, valor_p, ic95_inf e ic95_sup.
    """
    # Error estándar = raíz de la varianza: mide la incertidumbre de un coeficiente (más chico = más preciso).
    error = float(np.sqrt(max(varianza, 0.0)))
    distribucion = stats.t(gl) if np.isfinite(gl) else stats.norm()
    # Valor p: probabilidad de ver un resultado así de grande si NO hubiera efecto real.
    valor_p = float(2 * distribucion.sf(abs(beta / error))) if error > 0 else np.nan
    # Valor crítico para el intervalo de confianza del 95 %.
    critico = float(distribucion.ppf(0.975))
    return {"coef": float(beta), "se": error, "valor_p": valor_p, "ic95_inf": beta - critico * error, "ic95_sup": beta + critico * error}


def efecto_neto(inf: dict, nombres: list) -> dict:
    """
    Esta función calcula la suma de varios coeficientes (por ejemplo, día actual + rezagos) con su error estándar,
    usando la matriz de varianzas del modelo.

    :param inf:     el resultado de inferencia().
    :param nombres: los nombres de los coeficientes que se suman.
    :return:        un diccionario con la suma, su error estándar, el valor p y el intervalo de confianza del 95 %.
    """
    # Se ubican en la matriz los coeficientes que se van a sumar.
    posiciones = [inf["nombres"].index(n) for n in nombres]
    # La suma de los coeficientes (hoy + rezago 1-3 + rezago 4-6) es el "efecto neto" de una promoción sostenida.
    suma = inf["beta"][posiciones].sum()
    # La varianza de una suma incluye cómo se relacionan los coeficientes entre sí, no solo sus varianzas por separado.
    varianza = inf["vcov"][np.ix_(posiciones, posiciones)].sum()
    return resumen_coeficiente(suma, varianza, inf["gl"])

## 2. Lectura de los insumos

In [4]:
# Leer el panel, las exposiciones, el calendario y las tablas diarias de exposición (fechas con parse_dates)
# El "panel": una fila por producto y por día, con sus ventas (unit_sales_ppml) y si estaba en promoción (onpromotion).
panel = pd.read_parquet(
    archivo_panel, columns=["date", "item_nbr", "family", "unit_sales_ppml", "onpromotion"]
)
# Se ajustan los tipos de datos (texto/número) para que los cálculos funcionen.
panel["family"] = panel["family"].astype(str)
panel["onpromotion"] = panel["onpromotion"].astype(float)
panel["unit_sales_ppml"] = panel["unit_sales_ppml"].astype("float64")

# Las "exposiciones" del notebook 03: proporción de productos parecidos en promoción, por producto y día, y sus rezagos.
exposiciones = pd.read_parquet(archivo_exposiciones)
# El calendario (feriados, días de pago, el terremoto). parse_dates convierte la fecha en fecha de verdad.
calendario = pd.read_csv(archivo_calendario, parse_dates=["date"])
calendario[variables_calendario] = calendario[variables_calendario].astype(float)
for columna in ["dow", "semana", "mes"]:
    calendario[columna] = calendario[columna].astype(str)

# Solo las columnas del núcleo (y sus bloques), no las 22 categorías fuente
# Se arma la lista de las 30 columnas del núcleo: 10 categorías x 3 versiones (hoy, rezago 1-3, rezago 4-6).
columnas_nucleo = [columna_exposicion(k) + s for k in nucleo_exposicion for s in sufijos_bloques]
# Intensidad promocional diaria de cada categoría: una fila por día.
categorias_dia = pd.read_csv(archivo_categorias, parse_dates=["date"])
# Seguro: si falta alguna columna, el notebook se detiene con un mensaje.
faltantes = [c for c in columnas_nucleo if c not in categorias_dia.columns]
assert not faltantes, f"faltan columnas del núcleo en la tabla de categorías: {faltantes}"
categorias_dia = categorias_dia[["date"] + columnas_nucleo]

# La variable de control p__RESTO: promoción promedio de las categorías que no son del núcleo, calculada por separado para cada categoría objetivo.
resto_dia = pd.read_csv(archivo_resto, parse_dates=["date"])
columnas_resto = ["p__RESTO" + s for s in sufijos_bloques]
resto_dia = resto_dia[["family", "date"] + columnas_resto]

# Verificar las cifras de control y la unicidad de las claves de cada unión
# Controles de cordura: un "assert" es una afirmación que Python verifica; si es falsa, se detiene con un mensaje.
assert len(panel) == 1_935_758 and len(exposiciones) == len(panel), "el panel o las exposiciones no coinciden"
assert len(calendario) == 590 and calendario["ventana_terremoto"].sum() == 46, "el calendario no coincide con lo esperado"
assert not categorias_dia.duplicated("date").any(), "la tabla de categorías debe tener una fila por día"
assert not resto_dia.duplicated(["family", "date"]).any(), "la tabla del resto debe tener una fila por día y categoría"
assert set(categorias_objetivo) <= set(resto_dia["family"]), "falta el resto de alguna categoría objetivo"
print(f"Núcleo: {len(nucleo_exposicion)} categorías | columnas de exposición: {len(columnas_nucleo)} | "
      f"filas del resto: {len(resto_dia):,}")

# Controles extra de cobertura de fechas y de tipo de dato (una unión puede fallar en silencio si las fechas no coinciden)
assert all(pd.api.types.is_datetime64_any_dtype(d["date"]) for d in [panel, calendario, categorias_dia, resto_dia]), "date debe ser fecha en todas las tablas"
fechas_calendario = set(calendario["date"])
assert set(panel["date"]) <= fechas_calendario, "el panel tiene fechas que no están en el calendario"
assert set(categorias_dia["date"]) == fechas_calendario, "la tabla de categorías no cubre exactamente las fechas del calendario"
for c_obj in categorias_objetivo:
    assert set(resto_dia.loc[resto_dia["family"] == c_obj, "date"]) == fechas_calendario, f"el resto de {c_obj} no cubre las fechas del calendario"
assert calendario.loc[calendario["ventana_terremoto"], "date"].nunique() == 46, "la ventana del terremoto debe tener 46 fechas"
assert not calendario[["dow", "semana", "mes"]].isin(["nan"]).any().any(), "hay NaN convertidos en el texto 'nan' en dow/semana/mes"
print(f"Semanas distintas: {calendario['semana'].nunique()} | meses distintos: {calendario['mes'].nunique()} | días de la semana: {calendario['dow'].nunique()}")


Núcleo: 10 categorías | columnas de exposición: 30 | filas del resto: 18,880


Semanas distintas: 86 | meses distintos: 20 | días de la semana: 7


**Cómo leer esta salida.** `Núcleo: 10 categorías | columnas de exposición: 30 | filas del resto: 18,880`:
- **10 categorías del núcleo:** las categorías cuya promoción se usa como explicación (aunque solo 6 sean objetivo).
- **30 columnas** = 10 categorías x 3 versiones (hoy, rezago 1 a 3, rezago 4 a 6).
- **18.880 filas del resto** = 590 días x 32 categorías (una fila por día y categoría).

No es un resultado, sino un control: si alguna cifra no coincidiera con lo esperado, el notebook se habría detenido con un mensaje.

## 3. Base de estimación y modelo

`armar_base` es la de la fase 1 más las uniones de las exposiciones diarias (por `date`) y del resto (por `date` y `family`). Cada unión se verifica: sin duplicar ni perder filas.

In [5]:
def regresores_fase2(categoria: str) -> dict:
    """
    Esta función arma las listas de regresores de la fase 2 para una categoría objetivo: los de la fase 1, los de las
    otras categorías del núcleo (sin la propia categoría) y los del resto.

    :param categoria:   el nombre de la categoría objetivo.
    :return:            un diccionario con las listas propios, cruzados y resto.
    """
    # Las 9 variables propias: 3 tipos de promoción x 3 momentos (hoy, rezago 1-3, rezago 4-6).
    propios = [v + s for v in variables_base for s in sufijos_bloques]
    # Las otras 9 categorías del núcleo: todas menos la propia categoría objetivo.
    otras = [k for k in nucleo_exposicion if k != categoria]
    # Las 27 variables cruzadas: 9 categorías x 3 momentos.
    cruzados = [columna_exposicion(k) + s for k in otras for s in sufijos_bloques]
    # Las 3 variables de control del "resto" (no se interpretan).
    resto = ["p__RESTO" + s for s in sufijos_bloques]
    # Seguro: la promoción de la propia categoría no puede estar entre las "otras": ya está en las variables propias.
    assert columna_exposicion(categoria) not in [c.split("_l")[0] for c in cruzados], \
        "la columna de la propia categoría no debe estar entre los regresores"
    return {"propios": propios, "cruzados": cruzados, "resto": resto, "otras": otras}


diagnostico_nulos = {}                       # por categoría: variables con más datos faltantes (se llena en armar_base)


def armar_base(categoria: str) -> tuple:
    """
    Esta función arma la base de estimación de la fase 2 para una categoría objetivo y cuenta las filas que saca
    cada exclusión.

    :param categoria:   el nombre de la categoría objetivo.
    :return:            una tupla (base, conteos, regresores).
    """
    reg = regresores_fase2(categoria)
    todos = reg["propios"] + reg["cruzados"] + reg["resto"]

    # Paso 1: filas del panel de la categoría
    # PASO 1: quedarnos solo con las filas de la categoría objetivo.
    base = panel.loc[panel["family"] == categoria].copy()
    conteos = {"categoria": categoria, "filas_panel": len(base)}
    assert len(base) > 0, f"la categoría {categoria} no existe en el panel"

    # Paso 2: exposiciones SKU-día (uno a uno)
    # PASO 2: pegar a cada fila su exposición propia y de clase (producto y fecha). "one_to_one" exige que cada fila encuentre exactamente una pareja.
    exp = exposiciones.loc[exposiciones["item_nbr"].isin(base["item_nbr"].unique())]
    base = base.merge(exp, on=["item_nbr", "date"], how="left", validate="one_to_one")
    assert len(base) == conteos["filas_panel"] and base["dia_programa"].notna().all(), "falló la unión con las exposiciones"

    # Paso 3: calendario por fecha (muchos a uno)
    # PASO 3: pegar el calendario por fecha ("many_to_one": muchos productos comparten una misma fecha).
    columnas_calendario = ["date", "dow", "semana", "mes", "ventana_terremoto"] + variables_calendario
    base = base.merge(calendario[columnas_calendario], on="date", how="left", validate="many_to_one")
    assert len(base) == conteos["filas_panel"] and base["ventana_terremoto"].notna().all(), "falló la unión con el calendario"

    # Paso 4: intensidad de las otras categorías del núcleo, sin la propia categoría (muchos a uno por fecha)
    # PASO 4: pegar la promoción de las otras categorías del núcleo, por fecha.
    cols_cruzadas = ["date"] + reg["cruzados"]
    base = base.merge(categorias_dia[cols_cruzadas], on="date", how="left", validate="many_to_one")
    assert len(base) == conteos["filas_panel"], "la unión con las categorías cambió el número de filas"

    # Paso 5: resto para la categoría objetivo (muchos a uno por fecha, con family = j)
    # PASO 5: pegar el control "resto" correspondiente a esta categoría objetivo, por fecha.
    resto_j = resto_dia.loc[resto_dia["family"] == categoria, ["date"] + reg["resto"]]
    assert len(resto_j) == len(calendario), "el resto de la categoría debe tener una fila por día"
    base = base.merge(resto_j, on="date", how="left", validate="many_to_one")
    assert len(base) == conteos["filas_panel"], "la unión con el resto cambió el número de filas"

    # Paso 6: exclusiones (terremoto, días de programa, nulos), con su conteo
    # PASO 6a: excluir los días del terremoto de 2016 (las ventas de esos días responden al terremoto, no a las promociones).
    antes = len(base)
    base = base.loc[~base["ventana_terremoto"]]
    conteos["excl_terremoto"] = antes - len(base)

    antes = len(base)
    # PASO 6b: en categorías con programa semanal se sacan sus días de programa (ahí casi todo está en promoción y no hay con qué comparar).
    if categoria in categorias_programa:
        base = base.loc[~base["dia_programa"]]
    conteos["excl_programa"] = antes - len(base)

    antes = len(base)
    # PASO 6c: excluir filas con datos faltantes (sobre todo los primeros días, donde no se pueden calcular los rezagos).
    columnas_modelo = ["unit_sales_ppml"] + todos + variables_calendario + ["semana", "dow", "mes"]
    diagnostico_nulos[categoria] = base[columnas_modelo].isna().sum().sort_values(ascending=False)   # nulos antes de excluir
    base = base.dropna(subset=columnas_modelo)
    conteos["excl_nulos"] = antes - len(base)

    # Verificaciones de la sección 10.1
    # Controles finales: las cuentas deben cuadrar, no puede haber un mismo producto-día repetido y las ventas no pueden ser negativas.
    conteos["filas_base"] = len(base)
    assert conteos["filas_base"] == (
        conteos["filas_panel"] - conteos["excl_terremoto"] - conteos["excl_programa"] - conteos["excl_nulos"]
    ), "las filas de la base no cuadran con las exclusiones"
    assert not base.duplicated(["item_nbr", "date"]).any(), "la base debe tener una fila por SKU-día"
    assert (base["unit_sales_ppml"] >= 0).all(), "PPML exige ventas no negativas"
    conteos["n_dias_base"] = base["date"].nunique()
    conteos["n_skus_base"] = base["item_nbr"].nunique()
    assert conteos["n_dias_base"] <= dias_maximos, "la base usa más de 544 días: falta una exclusión"
    if categoria in categorias_programa:
        assert conteos["excl_programa"] > 0 and conteos["n_dias_base"] < dias_maximos, \
            "una categoría con programa debe perder días de programa"
    else:
        # Control más estricto que "a lo sumo 544": una categoría sin programa debe usar exactamente los 532 días esperados
        assert conteos["n_dias_base"] == dias_esperados, f"{categoria}: usa {conteos['n_dias_base']} días y se esperaban {dias_esperados}"
    return base.reset_index(drop=True), conteos, reg


def estimar_fase2(base: pd.DataFrame, reg: dict):
    """
    Esta función estima el PPML con efectos fijos de la fase 2.

    :param base:    la base SKU-día de una categoría, de armar_base.
    :param reg:     las listas de regresores de regresores_fase2.
    :return:        el modelo ajustado de pyfixest (fepois).
    """
    # Todas las variables de promoción juntas: 9 + 27 + 3 = 39.
    regresores = reg["propios"] + reg["cruzados"] + reg["resto"]
    # La fórmula del modelo: ventas ~ promociones + calendario | efectos fijos. Es una regresión de Poisson (PPML), que permite días con ventas cero y se lee en porcentajes.
    formula = "unit_sales_ppml ~ " + " + ".join(regresores + variables_calendario) + " | " + efectos_fijos
    with warnings.catch_warnings():
        # Se silencia el aviso de "singletons" (productos con una sola fila, que no aportan comparación y se descartan solos).
        warnings.filterwarnings("ignore", message=".*singleton.*")
        return pf.fepois(formula, data=base, vcov={"CRV1": agrupamiento})


def tabla_coeficientes(modelo, categoria: str, conteos: dict, reg: dict) -> pd.DataFrame:
    """
    Esta función ordena los coeficientes de un modelo de la fase 2 (error estándar, valor p, efecto en porcentaje) y
    agrega el efecto neto (día actual + dos bloques de rezagos) de cada variable de promoción.

    :param modelo:      el modelo ajustado de la fase 2.
    :param categoria:   el nombre de la categoría objetivo.
    :param conteos:     los conteos de armar_base más los tamaños del modelo.
    :param reg:         las listas de regresores.
    :return:            un DataFrame con una fila por coeficiente y por efecto neto.
    """
    # Errores estándar corregidos (ver función `inferencia`).
    inf = inferencia(modelo)
    conteos["vcov_corregida"] = inf["corregida"]
    # Una fila por variable, con coeficiente, error estándar, valor p e intervalo de confianza.
    tabla = pd.DataFrame([
        {"variable": n, **resumen_coeficiente(inf["beta"][i], inf["vcov"][i, i], inf["gl"])}
        for i, n in enumerate(inf["nombres"])
    ])
    # También se guardan el error estándar y el valor p "sin corregir" (tal como los entrega pyfixest), para comparar.
    tabla["se_sin_corregir"] = tabla["variable"].map(modelo.se())            # error estándar tal como lo entrega pyfixest
    tabla["valor_p_sin_corregir"] = tabla["variable"].map(modelo.pvalue())
    perdidas = [v for v in reg["propios"] + reg["cruzados"] + reg["resto"] if v not in set(tabla["variable"])]
    assert not perdidas, f"{categoria}: pyfixest eliminó variables por colinealidad: {perdidas}"

    # Efectos netos de cada grupo de promoción (día actual + l1_3 + l4_6)
    # Efecto "neto" de cada promoción: hoy + rezago 1-3 + rezago 4-6, es decir, lo que se asocia a mantener esa promoción durante una semana.
    grupos = variables_base + [columna_exposicion(k) for k in reg["otras"]] + ["p__RESTO"]
    netos = []
    for grupo in grupos:
        nombres = [grupo + s for s in sufijos_bloques]
        netos.append({"variable": f"NETO {grupo} (actual + l1_3 + l4_6)", **efecto_neto(inf, nombres)})
    tabla = pd.concat([tabla, pd.DataFrame(netos)], ignore_index=True)

    # Se traduce cada coeficiente a porcentaje y se calculan sus intervalos de confianza.
    tabla["efecto_pct"] = a_porcentaje(tabla["coef"])
    tabla["efecto_pct_ic95_inf"] = a_porcentaje(tabla["ic95_inf"])
    tabla["efecto_pct_ic95_sup"] = a_porcentaje(tabla["ic95_sup"])
    # Las variables de proporción (0 a 1) tienen un coeficiente por "pasar de 0 % a 100 %", un cambio irreal; por eso se agrega el efecto de +10 puntos porcentuales.
    es_proporcion = ~tabla["variable"].str.contains("onpromotion") & ~tabla["variable"].isin(variables_calendario)
    tabla["efecto_pct_10pp"] = np.where(es_proporcion, a_porcentaje(tabla["coef"], cambio_pp), np.nan)
    tabla.insert(0, "categoria", categoria)
    tabla["n_obs"] = conteos["n_obs_modelo"]
    tabla["n_skus"] = conteos["n_skus_modelo"]
    tabla["n_dias"] = conteos["n_dias_base"]
    return tabla

## 4. Estimación de las categorías objetivo

Se estiman, una por una, las **6 categorías sin programa semanal**. Cada categoría tiene su propio modelo; lo que cambia entre ellas es qué productos y días entran.

In [6]:
# Estimar cada categoría objetivo; la primera es la categoría piloto
# Listas vacías donde se irán guardando los resultados de cada categoría.
tablas, filas_controles = [], []
# Se repite el mismo procedimiento para cada una de las 6 categorías objetivo.
for categoria in categorias_objetivo:
    # 1) Armar la base de datos de la categoría.
    base, conteos, reg = armar_base(categoria)
    # 2) Estimar el modelo.
    modelo = estimar_fase2(base, reg)
    # 3) Anotar cuántas observaciones usó realmente el modelo (puede descartar productos de una sola fila).
    conteos["n_obs_modelo"] = int(modelo._N)
    conteos["n_skus_modelo"] = int(modelo._data["item_nbr"].nunique())
    # "Convergió" = el cálculo (por aproximaciones sucesivas) llegó a una solución estable.
    conteos["convergio"] = bool(modelo._convergence)
    # Si no convergió, el notebook se detiene: esos resultados no serían confiables.
    assert conteos["convergio"], f"el modelo de {categoria} no convergió: avisar al profesor guía"
    # 4) Ordenar los resultados en una tabla.
    tabla = tabla_coeficientes(modelo, categoria, conteos, reg)
    # Seguro: no puede haber errores estándar nulos, vacíos o negativos.
    assert tabla["se"].notna().all() and (tabla["se"] > 0).all(), f"{categoria}: hay errores estándar nulos o NaN"
    tablas.append(tabla)
    filas_controles.append(conteos)
    nulos = diagnostico_nulos[categoria]
    print("   nulos por variable (antes de excluir, principales): " + ", ".join(f"{c} {n:,}" for c, n in nulos[nulos > 0].head(4).items()))
    print(f"{categoria:18s} filas base {conteos['filas_base']:>8,} | días {conteos['n_dias_base']:>3} | "
          f"regresores {len(reg['propios']) + len(reg['cruzados']) + len(reg['resto'])} | "
          f"vcov corregida: {conteos['vcov_corregida']}")

# Se juntan las tablas de las 6 categorías en una sola.
coeficientes = pd.concat(tablas, ignore_index=True)

   nulos por variable (antes de excluir, principales): promo_clase_otros_l4_6 7,829, promo_cat_otras_clases_l4_6 6,140, p__DELI_l4_6 5,173, p__MEATS_l4_6 5,173
BEVERAGES          filas base  258,915 | días 532 | regresores 39 | vcov corregida: True


   nulos por variable (antes de excluir, principales): promo_clase_otros_l4_6 17,665, promo_cat_otras_clases_l4_6 13,044, p__DELI_l4_6 11,633, p__MEATS_l4_6 11,633
GROCERY I          filas base  556,533 | días 532 | regresores 39 | vcov corregida: True


   nulos por variable (antes de excluir, principales): promo_clase_otros_l4_6 5,895, promo_cat_otras_clases_l4_6 4,560, p__DELI_l4_6 4,105, p__MEATS_l4_6 4,105
CLEANING           filas base  197,454 | días 532 | regresores 39 | vcov corregida: True


   nulos por variable (antes de excluir, principales): promo_clase_otros_l4_6 3,671, promo_cat_otras_clases_l4_6 2,557, promo_clase_otros_l1_3 2,405, p__DELI_l4_6 2,287
DAIRY              filas base  109,779 | días 532 | regresores 39 | vcov corregida: True


   nulos por variable (antes de excluir, principales): promo_clase_otros_l4_6 1,560, promo_cat_otras_clases_l4_6 1,248, p__DELI_l4_6 1,168, p__MEATS_l4_6 1,168
BREAD/BAKERY       filas base   54,466 | días 532 | regresores 39 | vcov corregida: True


   nulos por variable (antes de excluir, principales): promo_clase_otros_l4_6 3,667, promo_clase_otros_l1_3 3,421, promo_clase_otros 3,164, promo_cat_otras_clases_l4_6 595
LIQUOR,WINE,BEER   filas base   21,466 | días 532 | regresores 39 | vcov corregida: True


**Cómo leer estas salidas (dos líneas por categoría).** Por ejemplo, para BEVERAGES:
- `nulos por variable (antes de excluir, principales): promo_clase_otros_l4_6 7,829, ...` → cuántas filas tenían datos faltantes en cada variable antes de excluirlas. Faltan sobre todo en los rezagos de 4 a 6 días, porque para calcularlos hacen falta 6 días anteriores (no existen al inicio del panel ni justo después del terremoto). **Esta línea la tomé del notebook de la compañera: sirve para entender por qué se pierden filas.**
- `BEVERAGES filas base 258,915 | días 532 | regresores 39 | vcov corregida: True`:
  - **filas base:** observaciones (un producto en un día) usadas para estimar.
  - **días 532:** días distintos con datos. Es exactamente lo esperado: 590 días con operación − 46 del terremoto − 12 sin historia de rezagos. Si fuera otro número, el notebook se detendría.
  - **regresores 39:** 9 propios + 27 cruzados (9 categorías x 3 momentos) + 3 del "resto".
  - **vcov corregida: True:** hubo que corregir la matriz de varianzas (detalle técnico; ver la guía). Pasa en las 6 categorías.

Si en alguna categoría aparece el aviso `observations removed because of separation`, significa que `pyfixest` sacó observaciones donde las ventas son siempre cero para ciertas combinaciones y el modelo no puede estimar ahí; es normal en categorías pequeñas. **En esta corrida no apareció ninguno.** Tampoco hace falta mirar el aviso de `sqrt`: ya está silenciado porque el código lo corrige.


## 5. Efectos cruzados $L_{jk}$ y corrección de Benjamini‑Hochberg

Para cada par $(j,k)$ se toma el coeficiente contemporáneo $\beta_{jk,0}$ de `p__<k>`. Se corrige por comparaciones múltiples sobre **todos** esos efectos a la vez. Se clasifica como **sinergia** ($\beta>0$ y significativo tras la corrección), **canibalización** ($\beta<0$ y significativo) o **sin efecto** (no se distingue de cero).

In [7]:
# Armar la tabla de efectos cruzados contemporáneos
# Se arma la tabla de los 54 pares (categoría afectada j, categoría en promoción k).
filas = []
# Para cada categoría afectada...
for categoria in categorias_objetivo:
    t = coeficientes.loc[coeficientes["categoria"] == categoria].set_index("variable")
    # ...y para cada otra categoría en promoción.
    for k in nucleo_exposicion:
        if k == categoria:
            continue
        columna = columna_exposicion(k)
        # fila = efecto de hoy (el que se interpreta); neto = hoy + rezagos (efecto de una promoción sostenida).
        fila, neto = t.loc[columna], t.loc[f"NETO {columna} (actual + l1_3 + l4_6)"]
        filas.append({
            "categoria_afectada": categoria, "categoria_en_promocion": k,
            "programa_afectada": categoria in categorias_programa, "programa_en_promocion": k in categorias_programa,
            "coef": fila["coef"], "se": fila["se"], "valor_p": fila["valor_p"],
            "se_sin_corregir": fila["se_sin_corregir"], "valor_p_sin_corregir": fila["valor_p_sin_corregir"],
            # L_jk = e^(0,1 x coef) - 1: cuánto cambian las ventas de j cuando 10 puntos porcentuales más del surtido de k están en promoción.
            "L_jk": np.exp(cambio_pp * fila["coef"]) - 1,
            "L_jk_ic95_inf": np.exp(cambio_pp * fila["ic95_inf"]) - 1,
            "L_jk_ic95_sup": np.exp(cambio_pp * fila["ic95_sup"]) - 1,
            "coef_neto": neto["coef"], "valor_p_neto": neto["valor_p"],
            "L_jk_neto": np.exp(cambio_pp * neto["coef"]) - 1,
        })
cruzados = pd.DataFrame(filas)

# Corrección de Benjamini-Hochberg sobre todos los efectos cruzados contemporáneos
# Seguro: deben ser exactamente 54 pares (6 categorías x 9 otras).
esperados = sum(len([k for k in nucleo_exposicion if k != j]) for j in categorias_objetivo)
assert len(cruzados) == esperados, f"se esperaban {esperados} efectos cruzados y hay {len(cruzados)}"
# Corrección de Benjamini-Hochberg sobre los 54 pares a la vez. `q` = valor p ajustado; `rechaza` = True si sigue siendo significativo con la nueva vara más exigente.
rechaza, q, _, _ = multipletests(cruzados["valor_p"].to_numpy(), alpha=tasa_fdr, method="fdr_bh")
cruzados["valor_q"] = q
cruzados["significativo_bh"] = rechaza
# Clasificación final: sinergia (coef > 0 y significativo), canibalización (coef < 0 y significativo) o sin efecto.
cruzados["clasificacion"] = np.where(
    rechaza & (cruzados["coef"] > 0), "sinergia", np.where(rechaza & (cruzados["coef"] < 0), "canibalizacion", "sin efecto")
)
cruzados["L_jk_pct"] = 100 * cruzados["L_jk"]
# Efecto de +10 puntos porcentuales puede ser un cambio grande frente a lo que varía de verdad la promoción de k:
# se agrega también el efecto de +1 desviación estándar (un cambio "típico" de la promoción de k).
desviacion = categorias_dia[[columna_exposicion(k) for k in nucleo_exposicion]].std()
cruzados["desv_est_promocion_k"] = cruzados["categoria_en_promocion"].map({k: desviacion[columna_exposicion(k)] for k in nucleo_exposicion})
cruzados["L_jk_1desv_pct"] = 100 * (np.exp(cruzados["desv_est_promocion_k"] * cruzados["coef"]) - 1)
# Resumen: cuántos de los 54 pares quedan significativos y de qué tipo.
print(f"Efectos cruzados: {len(cruzados)} | significativos tras BH: {int(rechaza.sum())}")
print(cruzados["clasificacion"].value_counts().to_string())

# Sensibilidad: lo mismo con los errores estándar tal como los entrega pyfixest (sin corregir la matriz de varianzas)
assert cruzados["valor_p_sin_corregir"].notna().all(), "hay valores p sin corregir nulos"
# Sensibilidad: lo mismo, pero con los errores estándar sin corregir. Sirve para ver si la conclusión depende de la corrección.
rechaza_sc, q_sc, _, _ = multipletests(cruzados["valor_p_sin_corregir"].to_numpy(), alpha=tasa_fdr, method="fdr_bh")
cruzados["valor_q_sin_corregir"] = q_sc
cruzados["significativo_bh_sin_corregir"] = rechaza_sc
cruzados["clasificacion_sin_corregir"] = np.where(
    rechaza_sc & (cruzados["coef"] > 0), "sinergia", np.where(rechaza_sc & (cruzados["coef"] < 0), "canibalizacion", "sin efecto")
)
print(f"\nSensibilidad (errores estándar sin corregir): significativos tras BH = {int(rechaza_sc.sum())}")
print(cruzados.loc[rechaza_sc, ["categoria_afectada", "categoria_en_promocion", "coef", "valor_p_sin_corregir", "valor_q_sin_corregir"]].round(4).to_string(index=False))

Efectos cruzados: 54 | significativos tras BH: 0
clasificacion
sin efecto    54

Sensibilidad (errores estándar sin corregir): significativos tras BH = 3
categoria_afectada categoria_en_promocion    coef  valor_p_sin_corregir  valor_q_sin_corregir
  LIQUOR,WINE,BEER                  MEATS  0.1980                0.0003                0.0144
  LIQUOR,WINE,BEER                POULTRY -1.1481                0.0019                0.0341
  LIQUOR,WINE,BEER              GROCERY I  2.2977                0.0013                0.0341


**Cómo leer esta salida (efectos cruzados y Benjamini‑Hochberg).**

1. `Efectos cruzados: 54 | significativos tras BH: 0` → de los **54 pares** (6 categorías afectadas x 9 categorías en promoción), **ninguno** sobrevive a la corrección de Benjamini‑Hochberg con los errores estándar corregidos. La tabla (`sin efecto 54`) dice lo mismo.
2. **Por qué cero no es un fracaso:** con 54 pares y un criterio de 5 %, ya se esperan unos 2 o 3 "significativos" solo por suerte (el "billete de lotería"). Si todos los que aparecen son compatibles con la suerte, la conclusión honesta es: **con estos datos no se puede asegurar que la promoción de una categoría cambie las ventas de otra**.
3. **Sensibilidad con errores estándar sin corregir:** `significativos tras BH = 3`. Son los **mismos 3 pares que obtiene la compañera en 06.5**, todos en LIQUOR,WINE,BEER: afectada por MEATS (+), POULTRY (−) y GROCERY I (+). Aparecen solo porque no se corrigió la matriz de varianzas; no conviene presentarlos como hallazgos firmes.

**Cómo se lee cada columna de la tabla de sensibilidad:**
- `coef`: coeficiente del modelo (en escala logarítmica, no es un %).
- `valor_p_sin_corregir`: probabilidad de ver ese resultado si no hubiera efecto. Más chico = más "llamativo".
- `valor_q_sin_corregir`: ese valor p ajustado por haber probado 54 pares. **El criterio es q < 0,05.**

**Un ejemplo de lectura:** `LIQUOR,WINE,BEER – MEATS: coef 0,198, valor p 0,0003, valor q 0,0144`. Cuando más surtido de carnes está en promoción, las ventas de licores suben. Con la versión sin corregir pasa el filtro (q = 0,0144 < 0,05); con la corregida no pasa. La decisión de cuál versión usar le corresponde al profesor guía.

**Cómo se traduce un coeficiente a porcentaje (con un ejemplo).** Tomemos `LIQUOR,WINE,BEER – GROCERY I: coef 2,2977`:
- Efecto de **+10 puntos porcentuales** del surtido de GROCERY I en promoción: e^(0,1 x 2,2977) − 1 = e^0,2298 − 1 = **+25,8 %** en las ventas de licores.
- Parece enorme, pero con la matriz corregida el valor p es 0,0029 y el q es 0,12 (no pasa el 0,05), y el error estándar también es grande. **Un coeficiente grande no es lo mismo que un efecto seguro.**

In [8]:
# Cifras que se citan en el texto: se calculan aquí para que no dependan de números escritos a mano
n = len(cruzados)
print(f"Pares con coeficiente negativo: {int((cruzados['coef'] < 0).sum())} de {n} | positivo: {int((cruzados['coef'] > 0).sum())}")
print(f"L_jk (por +10 pp): promedio {cruzados['L_jk_pct'].mean():+.2f} % | mediana {cruzados['L_jk_pct'].median():+.2f} %")
print(f"Pares con |L_jk| menor de 10 %: {100 * (cruzados['L_jk_pct'].abs() < 10).mean():.1f} %")
print(f"Pares con valor p < 0,05 (matriz corregida, antes de Benjamini-Hochberg): {int((cruzados['valor_p'] < 0.05).sum())} de {n}")
print("\nPromedio de L_jk según la categoría EN PROMOCIÓN (columnas de la matriz):")
print(cruzados.groupby("categoria_en_promocion")["L_jk_pct"].mean().round(1).sort_values().to_string())
print("\nLos 5 pares con mayor |L_jk|: efecto por +10 pp frente a +1 desviación estándar de la promoción de k:")
grandes = cruzados.reindex(cruzados["L_jk_pct"].abs().sort_values(ascending=False).index).head(5)
print(grandes[["categoria_afectada", "categoria_en_promocion", "desv_est_promocion_k", "L_jk_pct", "L_jk_1desv_pct", "valor_q"]].round(3).to_string(index=False))

Pares con coeficiente negativo: 32 de 54 | positivo: 22
L_jk (por +10 pp): promedio -1.54 % | mediana -2.02 %
Pares con |L_jk| menor de 10 %: 96.3 %
Pares con valor p < 0,05 (matriz corregida, antes de Benjamini-Hochberg): 10 de 54

Promedio de L_jk según la categoría EN PROMOCIÓN (columnas de la matriz):
categoria_en_promocion
CLEANING           -6.3
BREAD/BAKERY       -5.2
PRODUCE            -5.2
BEVERAGES          -4.6
LIQUOR,WINE,BEER   -1.6
DELI               -1.2
POULTRY            -0.3
DAIRY               0.7
MEATS               1.0
GROCERY I           7.3

Los 5 pares con mayor |L_jk|: efecto por +10 pp frente a +1 desviación estándar de la promoción de k:
categoria_afectada categoria_en_promocion  desv_est_promocion_k  L_jk_pct  L_jk_1desv_pct  valor_q
  LIQUOR,WINE,BEER              GROCERY I                 0.030    25.831           7.204    0.122
  LIQUOR,WINE,BEER                POULTRY                 0.032   -10.846          -3.614    0.122
         BEVERAGES           B

**Cómo leer esta salida (las cifras que se citan en el texto).** Se calculan aquí para que cada número del texto salga del código:
- **32 pares negativos y 22 positivos:** hay un poco más de canibalización que de sinergia, pero la diferencia no es grande.
- **L_jk promedio de −1,54 % y mediana −2,02 %:** por cada +10 puntos de promoción en otra categoría, las ventas cambian, en promedio, algo más de 1 %.
- **96,3 % de los pares con |L| menor de 10 %:** casi todos los efectos son pequeños.
- **10 de 54 con p < 0,05 antes de la corrección por comparaciones múltiples:** con 54 pruebas se esperarían ~2 o 3 por azar.
- **Promedio por categoría en promoción:** Limpieza (−6,3 %), Panadería (−5,2 %), Frutas y verduras (−5,2 %) y Bebidas (−4,6 %) tienden a restar ventas a las demás; Abarrotes (+7,3 %) a sumar, pero está empujado por un solo caso (licores).
- **+10 puntos vs +1 desviación estándar:** el +25,8 % de licores–abarrotes es por +10 puntos, pero la promoción de abarrotes casi nunca se mueve tanto (su desviación estándar es de solo 0,03 = 3 puntos; su máximo es 18 puntos). Con un cambio "típico" (+1 desviación estándar) el efecto es **+7,2 %**. Es mejor presentar los dos, para no dar la impresión de un efecto enorme.

## 6. Controles de verificación (sección 10.1)

- Una fila por SKU‑día, días usados exactamente 532 y conteos de exclusiones: ya verificados con `assert` en `armar_base`.
- **Tamaño de los coeficientes:** ningún coeficiente de promoción (hoy y rezagos) mayor que 5 en valor absoluto. **Ojo:** el efecto propio (una variable 0/1) y los cruzados (proporciones de 0 a 1) **no están en las mismas unidades**; por eso se compara la mediana de los cruzados con el propio, y no el máximo.
- **Errores estándar:** si casi todos los efectos cruzados tuvieran p < 0,001, probablemente no se habría agrupado por semana.
- **Estabilidad:** el efecto propio de la fase 2 debería parecerse al de la fase 1.

In [9]:
# Controles de tamaño, de errores estándar y de estabilidad respecto de la fase 1
# Se lee el efecto propio estimado en la fase 1 (notebook 05) para compararlo.
fase1 = pd.read_csv(archivo_fase1).query("variable == 'onpromotion'").set_index("categoria")["coef"]
assert fase1.index.is_unique, "la fase 1 tiene más de una fila onpromotion por categoría"
# Se parte de los conteos de cada categoría.
controles = pd.DataFrame(filas_controles)
propio_fase2 = coeficientes.query("variable == 'onpromotion'").set_index("categoria")["coef"]
controles["coef_propio_fase2"] = controles["categoria"].map(propio_fase2)
controles["coef_propio_fase1"] = controles["categoria"].map(fase1)
# Estabilidad: el efecto propio no debería cambiar al agregar las otras categorías. Diferencia cercana a 0 = estable.
controles["dif_propio_fase2_menos_fase1"] = controles["coef_propio_fase2"] - controles["coef_propio_fase1"]
# Se agrupan los 54 pares por categoría afectada.
por_cat = cruzados.groupby("categoria_afectada")
# Tamaño: el coeficiente cruzado más grande y el típico (mediana). OJO: no son comparables en unidades con el propio (el propio es un 0/1; el cruzado es una proporción de 0 a 1), así que se compara sobre todo la mediana.
controles["max_abs_coef_cruzado"] = controles["categoria"].map(por_cat["coef"].apply(lambda x: x.abs().max()))
controles["mediana_abs_coef_cruzado"] = controles["categoria"].map(por_cat["coef"].apply(lambda x: x.abs().median()))
controles["coef_cruzado_mayor_que_5"] = controles["max_abs_coef_cruzado"] > umbral_coeficiente
# Cobertura completa: el control de "mayor que 5" se aplica a TODOS los coeficientes de promoción (con rezagos), no solo a los 54 de hoy.
vars_promo = coeficientes.loc[~coeficientes["variable"].str.startswith("NETO") & ~coeficientes["variable"].isin(variables_calendario)]
controles["max_abs_coef_promo_todos"] = controles["categoria"].map(vars_promo.groupby("categoria")["coef"].apply(lambda x: x.abs().max()))
controles["coef_promo_mayor_que_5"] = controles["max_abs_coef_promo_todos"] > umbral_coeficiente
# Alarma de errores estándar: si casi todos los pares tuvieran p < 0,001, probablemente no se habría agrupado bien.
controles["pct_cruzados_p_menor_0_001"] = controles["categoria"].map(por_cat["valor_p"].apply(lambda x: 100 * (x < 0.001).mean()))
controles["pct_cruzados_p_menor_0_05"] = controles["categoria"].map(por_cat["valor_p"].apply(lambda x: 100 * (x < 0.05).mean()))
controles["propio_positivo"] = controles["coef_propio_fase2"] > 0
columnas_mostrar = [
    "categoria", "filas_panel", "excl_terremoto", "excl_programa", "excl_nulos", "filas_base", "n_obs_modelo", "n_skus_modelo",
    "n_dias_base", "coef_propio_fase1", "coef_propio_fase2", "mediana_abs_coef_cruzado", "max_abs_coef_cruzado", "max_abs_coef_promo_todos", "pct_cruzados_p_menor_0_001",
    "pct_cruzados_p_menor_0_05", "vcov_corregida",
]
print(controles[columnas_mostrar].round(3).to_string(index=False))

# Si algún control falla, el notebook se detiene con un mensaje.
assert not controles["coef_cruzado_mayor_que_5"].any() and not controles["coef_promo_mayor_que_5"].any(), "hay coeficientes de promoción mayores que 5 en valor absoluto: revisar colinealidad"
assert (controles["pct_cruzados_p_menor_0_001"] < 50).all(), "casi todos los efectos cruzados son muy significativos: revisar el agrupamiento"
print("\nCategorías con efecto propio no positivo en la fase 2:", controles.loc[~controles["propio_positivo"], "categoria"].tolist() or "ninguna")

       categoria  filas_panel  excl_terremoto  excl_programa  excl_nulos  filas_base  n_obs_modelo  n_skus_modelo  n_dias_base  coef_propio_fase1  coef_propio_fase2  mediana_abs_coef_cruzado  max_abs_coef_cruzado  max_abs_coef_promo_todos  pct_cruzados_p_menor_0_001  pct_cruzados_p_menor_0_05  vcov_corregida
       BEVERAGES       287407           20663              0        7829      258915        258914            575          532              0.469              0.464                     0.344                 0.939                     2.187                         0.0                     22.222            True
       GROCERY I       620919           46701              0       17685      556533        556531           1151          532              0.473              0.472                     0.226                 0.751                     2.044                         0.0                     33.333            True
        CLEANING       220098           16748              0        58

**Cómo leer esta tabla (controles).** Cada fila es una categoría:
- `filas_panel → filas_base`: cuántas filas había y cuántas quedaron tras sacar el terremoto (`excl_terremoto`), los días de programa (`excl_programa`, 0 porque ninguna tiene programa) y los datos faltantes (`excl_nulos`). Las cuentas cuadran.
- `n_obs_modelo` vs `filas_base`: casi iguales; la diferencia (como mucho 1 o 2 filas) son productos con una sola observación que el modelo no puede usar.
- `n_dias_base`: **532 en las 6**, exactamente lo esperado.
- `coef_propio_fase1` vs `coef_propio_fase2`: el efecto propio **casi no cambia** al agregar las otras categorías (por ejemplo, BEVERAGES 0,469 vs 0,464). Es bueno: el modelo es estable.
- `mediana_abs_coef_cruzado`: el coeficiente cruzado "típico" (entre 0,23 y 0,48): **menor** que el efecto propio (0,35 a 0,64) en las 6 categorías, salvo en las que se acercan; es la comparación razonable.
- `max_abs_coef_cruzado`: el cruzado más grande (entre 0,6 y 2,3). **Ojo:** en las 6 categorías este máximo es **mayor** que el efecto propio (por ejemplo, 0,94 frente a 0,46). Por eso la frase "los cruzados son mucho menores que el propio" **no es cierta tal cual**; hay que decirlo así: el cruzado típico es menor, pero hay valores extremos, y sus unidades no son comparables (el propio es un 0/1; el cruzado, una proporción).
- `max_abs_coef_promo_todos`: el mayor coeficiente de promoción **incluyendo los rezagos** (entre 1,5 y 4,2): sigue bajo el umbral de 5, pero el 4,2 de LIQUOR,WINE,BEER está cerca y es una categoría de solo 49 SKUs, así que sus coeficientes son muy inestables.
- `pct_cruzados_p_menor_0_001`: 0 en todas → ningún par es "extremadamente significativo", lo que descarta que los errores estándar estén mal calculados.
- `pct_cruzados_p_menor_0_05`: entre 0 % y 33 % de los pares tienen p < 0,05 **antes** de corregir por comparaciones múltiples (10 de 54 en total).
- **"Categorías con efecto propio no positivo: ninguna"** → en las 6, la promoción propia aumenta las ventas, como se espera. (En la versión de 10 categorías, POULTRY y PRODUCE salían negativas por el programa semanal; por eso no están aquí.)

### Verificación independiente con otro programa

Para descartar que el resultado dependa de `pyfixest`, se vuelve a estimar un modelo de la fase 2 con `statsmodels` (regresión de Poisson con variables binarias por SKU, día de la semana y mes) en una categoría pequeña. Los coeficientes deben coincidir con los de `pyfixest`.

In [10]:
import statsmodels.api as sm

# Categoría chica (pocos productos) para que la verificación sea rápida.
categoria_control = "LIQUOR,WINE,BEER"
base_c, conteos_c, reg_c = armar_base(categoria_control)
modelo_c = estimar_fase2(base_c, reg_c)
regresores_c = reg_c["propios"] + reg_c["cruzados"] + reg_c["resto"] + variables_calendario

# Quitar los SKUs que pyfixest elimina (sin variación) para comparar con la misma muestra
# Se sacan los productos que nunca vendieron: no aportan y pyfixest también los descarta.
ventas_por_sku = base_c.groupby("item_nbr")["unit_sales_ppml"].transform("sum")
base_c = base_c.loc[ventas_por_sku > 0]
# VERIFICACIÓN INDEPENDIENTE: se recrean los efectos fijos con variables 0/1 (el método "de toda la vida")...
dummies = pd.get_dummies(base_c[["item_nbr", "dow", "mes"]].astype(str), drop_first=True, dtype=float)
X = pd.concat([base_c[regresores_c].astype(float), dummies], axis=1)
X = sm.add_constant(X)
# ...y se estima el mismo modelo con otro programa (statsmodels).
glm = sm.GLM(base_c["unit_sales_ppml"].astype(float), X, family=sm.families.Poisson()).fit(maxiter=100)
# Se comparan los coeficientes de los dos programas.
comparacion = pd.DataFrame({"pyfixest": modelo_c.coef(), "statsmodels": glm.params.reindex(modelo_c.coef().index)})
comparacion["dif_abs"] = (comparacion["pyfixest"] - comparacion["statsmodels"]).abs()
print(f"{categoria_control}: máxima diferencia absoluta entre programas = {comparacion['dif_abs'].max():.2e}")
# Si difieren más de 0,001, el notebook se detiene.
assert comparacion["dif_abs"].max() < 1e-3, "los coeficientes de pyfixest y statsmodels no coinciden"

LIQUOR,WINE,BEER: máxima diferencia absoluta entre programas = 4.91e-11


**Cómo leer esta salida.** `máxima diferencia absoluta entre programas = 4.91e-11`: se estimó el mismo modelo con dos programas distintos y la mayor diferencia entre sus coeficientes es prácticamente cero. Los dos dan lo mismo, así que el resultado no depende de una peculiaridad de `pyfixest`.

**Un límite de esta verificación:** compara los **coeficientes**, pero no los **errores estándar** (que es justo lo más discutible, por la matriz de varianzas). La robustez de los errores estándar se revisa en la sección 8 (tres formas de calcularlos).


## 7. Exportación

In [11]:
# Guardar las tablas en archivos nuevos dentro de resultados/tablas
# Se guardan las tablas en archivos CSV (se abren con Excel) en resultados/tablas.
coeficientes.to_csv(os.path.join(ruta_tablas, "tabla_28_fase2c_coeficientes.csv"), index=False)
cruzados.to_csv(os.path.join(ruta_tablas, "tabla_29_fase2c_efectos_cruzados.csv"), index=False)
controles.to_csv(os.path.join(ruta_tablas, "tabla_30_fase2c_bases_y_controles.csv"), index=False)

# Se muestran los 15 pares con valor p más pequeño (los "más sospechosos de ser reales").
resumen = cruzados.sort_values("valor_p").head(15)[
    ["categoria_afectada", "categoria_en_promocion", "coef", "L_jk_pct", "valor_p", "valor_q", "clasificacion"]
]
print(resumen.round(4).to_string(index=False))
print(">>>> Proceso finalizado <<<<")

categoria_afectada categoria_en_promocion    coef  L_jk_pct  valor_p  valor_q clasificacion
  LIQUOR,WINE,BEER              GROCERY I  2.2977   25.8312   0.0029   0.1222    sin efecto
         BEVERAGES           BREAD/BAKERY -0.9389   -8.9620   0.0067   0.1222    sin efecto
  LIQUOR,WINE,BEER                POULTRY -1.1481  -10.8461   0.0075   0.1222    sin efecto
          CLEANING                PRODUCE -0.8121   -7.8004   0.0090   0.1222    sin efecto
         GROCERY I                   DELI -0.2952   -2.9084   0.0180   0.1940    sin efecto
         BEVERAGES               CLEANING -0.9274   -8.8569   0.0256   0.2269    sin efecto
         GROCERY I               CLEANING -0.7188   -6.9355   0.0346   0.2269    sin efecto
             DAIRY           BREAD/BAKERY -0.6196   -6.0082   0.0369   0.2269    sin efecto
             DAIRY               CLEANING -0.6446   -6.2426   0.0473   0.2269    sin efecto
         GROCERY I           BREAD/BAKERY -0.6476   -6.2711   0.0480   0.2269   

**Cómo leer esta tabla (los 15 pares con menor valor p).** Con la versión **corregida**:
- `coef`: coeficiente del modelo.
- `L_jk_pct`: cambio de las ventas de la categoría afectada por +10 puntos porcentuales de promoción en la otra. Por ejemplo, `BEVERAGES – BREAD/BAKERY: −9,0 %` significa: cuando 10 puntos más del surtido de panadería están en promoción, las ventas de bebidas caen ~9 % (canibalización, pero no significativa tras la corrección).
- `valor_p`: va de 0,003 a 0,07 en estos 15.
- `valor_q`: va de 0,12 a 0,26. **Ninguno baja de 0,05**, por eso `clasificacion` dice `sin efecto` en todos.

**Conclusiones de la Fase 2 (6 categorías) en palabras simples:**
1. **No hay evidencia firme** de que la promoción de una categoría cambie las ventas de otra: 0 de 54 pares son significativos con la corrección.
2. Los efectos cruzados son **pequeños** (el 96 % está entre −10 % y +10 % por cada +10 puntos de promoción) y tienden a ser negativos: 32 de 54 negativos y 22 positivos, con un promedio de −1,5 %.
3. **Un patrón descriptivo** (no significativo): cuando **Limpieza**, **Panadería** o **Frutas y verduras** están en promoción, las demás categorías tienden a vender algo menos (promedio −5 % a −6 %); cuando lo está **Abarrotes**, algo más (+7 %, aunque empujado por el caso de licores).
4. Los 3 pares que aparecen sin corregir (todos de licores) dependen de **no** corregir la matriz de varianzas: con la matriz corregida y con agrupación solo por semana, **0 de 54** son significativos (sección 8). Se recomienda no presentarlos como resultados firmes. Además, licores es la categoría más frágil (49 SKUs, 12,6 % de filas descartadas por la clase; sección 9).
5. Es una asociación, no una causa, de una sola tienda en 2016‑2017; no se mide la rentabilidad.
6. **Pendiente del profesor guía:** (a) si se usa la corrección de la matriz de varianzas como versión principal; (b) la regla de los rezagos en las categorías con programa, que habilitaría estimarlas como objetivo.

## 8. Sensibilidad: errores estándar agrupados solo por semana

Hay tres formas de calcular la incertidumbre de los efectos cruzados, y la conclusión depende de cuál se use:
1. **Dos vías (producto y semana), con la matriz corregida** → la versión principal de este notebook.
2. **Dos vías, sin corregir la matriz** → la sensibilidad anterior.
3. **Solo por semana** → como los efectos cruzados varían solo de un día a otro (no entre productos), agrupar por semana es una opción natural y **siempre da una matriz válida**. Se calcula aquí.

In [12]:
# Re-estimar las 6 categorías agrupando los errores estándar solo por semana, y aplicar Benjamini-Hochberg a los 54 pares
filas_sem = []
for categoria in categorias_objetivo:
    base_s, _, reg_s = armar_base(categoria)
    regresores_s = reg_s["propios"] + reg_s["cruzados"] + reg_s["resto"] + variables_calendario
    formula_s = "unit_sales_ppml ~ " + " + ".join(regresores_s) + " | " + efectos_fijos
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*singleton.*")
        m_sem = pf.fepois(formula_s, data=base_s, vcov={"CRV1": "semana"})
    tidy = m_sem.tidy()
    for k in reg_s["otras"]:
        col = columna_exposicion(k)
        filas_sem.append({"categoria_afectada": categoria, "categoria_en_promocion": k, "coef": tidy.loc[col, "Estimate"],
                          "se": tidy.loc[col, "Std. Error"], "valor_p": tidy.loc[col, "Pr(>|t|)"]})
sens_sem = pd.DataFrame(filas_sem)
assert len(sens_sem) == len(cruzados) and sens_sem["valor_p"].notna().all(), "faltan pares en la sensibilidad por semana"
rech_sem, q_sem, _, _ = multipletests(sens_sem["valor_p"].to_numpy(), alpha=tasa_fdr, method="fdr_bh")
sens_sem["valor_q"] = q_sem
sens_sem["significativo_bh"] = rech_sem
sens_sem.to_csv(os.path.join(ruta_tablas, "tabla_32_fase2c_sensibilidad_solo_semana.csv"), index=False)

resumen_tres = pd.DataFrame({
    "forma_de_calcular_la_incertidumbre": ["1) dos vías, matriz corregida (principal)", "2) dos vías, sin corregir", "3) solo por semana"],
    "significativos_tras_BH_de_54": [int(cruzados["significativo_bh"].sum()), int(cruzados["significativo_bh_sin_corregir"].sum()), int(rech_sem.sum())],
    "menor_valor_q": [cruzados["valor_q"].min(), cruzados["valor_q_sin_corregir"].min(), sens_sem["valor_q"].min()],
})
print(resumen_tres.round(4).to_string(index=False))
print("\nLos 6 pares con menor valor p agrupando solo por semana:")
print(sens_sem.sort_values("valor_p").head(6).round(4).to_string(index=False))

       forma_de_calcular_la_incertidumbre  significativos_tras_BH_de_54  menor_valor_q
1) dos vías, matriz corregida (principal)                             0         0.1222
                2) dos vías, sin corregir                             3         0.0144
                       3) solo por semana                             0         0.0928

Los 6 pares con menor valor p agrupando solo por semana:
categoria_afectada categoria_en_promocion    coef     se  valor_p  valor_q  significativo_bh
         BEVERAGES           BREAD/BAKERY -0.9389 0.3078   0.0023   0.0928             False
  LIQUOR,WINE,BEER                POULTRY -1.1481 0.4067   0.0048   0.0928             False
          CLEANING                PRODUCE -0.8121 0.2904   0.0052   0.0928             False
  LIQUOR,WINE,BEER              GROCERY I  2.2977 0.9094   0.0115   0.1550             False
  LIQUOR,WINE,BEER                  MEATS  0.1980 0.0811   0.0146   0.1550             False
         GROCERY I                  

**Cómo leer esta salida (tres formas de calcular la incertidumbre).**

| Forma | Significativos de 54 | Menor valor q |
|---|---|---|
| 1) Dos vías, matriz corregida (principal) | **0** | 0,122 |
| 2) Dos vías, sin corregir | **3** | 0,014 |
| 3) Solo por semana | **0** | 0,093 |

- **Dos de las tres formas dan cero.** La única que da 3 es la que usa la matriz de varianzas sin corregir, que tiene valores propios negativos (es inválida). Eso refuerza la conclusión de que **no hay efectos cruzados firmes**.
- Con agrupación solo por semana, los pares "más llamativos" son otros que en la versión sin corregir (Bebidas–Panadería, Licores–Aves, Limpieza–Frutas y verduras), y ninguno pasa el 0,05 tras la corrección (el menor q es 0,093).
- Esta tercera forma **no está en la metodología**; se agrega como sensibilidad. La decisión de cuál es la principal le corresponde al profesor guía.

## 9. Datos faltantes "estructurales": SKUs sin otros productos en su clase

La variable `promo_clase_otros` (proporción de los OTROS productos de la clase en promoción) **no está definida** cuando la clase no tiene otros productos activos ese día. Esas filas se descartan en `armar_base` (el `dropna`), y eso cambia la muestra: no son datos "perdidos" por azar, sino productos que están solos en su clase. Aquí se mide cuántos.

In [13]:
# Contar, por categoría, las filas y los SKUs afectados por la falta de "otros productos en la clase" (rezago 0: no puede venir de la ventana de rezagos)
filas_nul = []
for categoria in categorias_objetivo:
    x = panel.loc[panel["family"] == categoria].merge(exposiciones[["item_nbr", "date", "promo_clase_otros"]], on=["item_nbr", "date"], how="left", validate="one_to_one")
    x = x.merge(calendario[["date", "ventana_terremoto"]], on="date", how="left", validate="many_to_one")
    x = x.loc[~x["ventana_terremoto"]]
    nulo = x["promo_clase_otros"].isna()
    frac_sku = nulo.groupby(x["item_nbr"]).mean()
    filas_nul.append({
        "categoria": categoria, "filas_sin_terremoto": len(x), "filas_con_clase_sin_otros": int(nulo.sum()),
        "pct_filas": 100 * nulo.mean(), "skus": int(x["item_nbr"].nunique()),
        "skus_con_algun_dia_sin_otros": int((frac_sku > 0).sum()), "skus_casi_siempre_solos_(>90%_de_sus_dias)": int((frac_sku > 0.9).sum()),
    })
nulos_estructurales = pd.DataFrame(filas_nul)
nulos_estructurales.to_csv(os.path.join(ruta_tablas, "tabla_33_fase2c_nulos_estructurales_clase.csv"), index=False)
print(nulos_estructurales.round(1).to_string(index=False))

       categoria  filas_sin_terremoto  filas_con_clase_sin_otros  pct_filas  skus  skus_con_algun_dia_sin_otros  skus_casi_siempre_solos_(>90%_de_sus_dias)
       BEVERAGES               266744                       1731        0.6   581                             5                                           4
       GROCERY I               574218                       4749        0.8  1175                            20                                           9
        CLEANING               203350                       1366        0.7   399                             6                                           3
           DAIRY               113450                       1138        1.0   229                             5                                           2
    BREAD/BAKERY                56026                        330        0.6   108                             3                                           3
LIQUOR,WINE,BEER                25148                       3164

**Cómo leer esta tabla (datos faltantes "estructurales").** Cada fila es una categoría:
- `filas_con_clase_sin_otros` y `pct_filas`: filas donde el producto está **solo en su clase** ese día, así que no se puede calcular "la proporción de los otros productos de la clase en promoción". Esas filas se descartan del modelo.
- `skus_con_algun_dia_sin_otros`: cuántos productos tienen al menos un día así.
- `skus_casi_siempre_solos`: productos que están solos en su clase en más del 90 % de sus días (salen casi completos del modelo).

**Qué se ve:** en 5 de las 6 categorías es un problema **pequeño** (0,6 % a 1,0 % de las filas). **LIQUOR,WINE,BEER es la excepción:** el **12,6 %** de las filas y **16 de 60 SKUs** (7 de ellos casi siempre solos en su clase) quedan fuera. Por eso el modelo de licores usa 49 SKUs y no 60.

**Por qué importa:** los resultados de licores (incluidos los tres pares que aparecen sin corregir) se estiman sobre una muestra distinta de la del resto, y quizá menos representativa. No es un error del código, sino una decisión que había que documentar. Una alternativa (poner 0 más un indicador de "sin otros") cambiaría el modelo, así que se deja para decidir con el profesor guía.

## 10. Comparación con la versión de 10 categorías (y con la de la compañera)

Aquí se comprueban dos cosas:
1. **Los coeficientes no cambian** al usar solo 6 categorías: cada categoría se estima por separado, así que debe dar lo mismo que en la versión de 10 (`tabla_11` y `tabla_12` del notebook 06 original).
2. **Lo que sí cambia es la corrección por comparaciones múltiples:** con 54 pruebas en vez de 90, el criterio de Benjamini‑Hochberg es menos exigente. Se muestra el resultado con y sin la corrección de la matriz de varianzas.

In [14]:
# Leer las tablas de la versión de 10 categorías (notebook 06 original); no se modifican
original_coef = pd.read_csv(os.path.join(ruta_tablas, "tabla_11_fase2_coeficientes.csv"))
original_cruz = pd.read_csv(os.path.join(ruta_tablas, "tabla_12_fase2_efectos_cruzados.csv"))

# 1) Coeficientes: comparar categoría por categoría y variable por variable (solo las 6 categorías de este notebook)
original_coef = original_coef.loc[original_coef["categoria"].isin(categorias_objetivo)]
union = original_coef.merge(coeficientes, on=["categoria", "variable"], suffixes=("_orig", "_nuevo"), validate="one_to_one")
assert len(union) == len(original_coef) == len(coeficientes), "las tablas de coeficientes no tienen las mismas filas"
dif_coef = (union["coef_orig"] - union["coef_nuevo"]).abs().max()
print(f"Máxima diferencia en coeficientes frente a la versión de 10 categorías: {dif_coef:.2e}")
assert dif_coef < 1e-4, "los coeficientes cambiaron: revisar"

# 2) Efectos cruzados: mismos 54 pares, mismos valores p; cambia solo la corrección por comparaciones múltiples
original_cruz = original_cruz.loc[original_cruz["categoria_afectada"].isin(categorias_objetivo)]
pares = original_cruz.merge(cruzados, on=["categoria_afectada", "categoria_en_promocion"], suffixes=("_10cat", "_6cat"), validate="one_to_one")
assert len(pares) == len(cruzados) == 54, "deben ser los mismos 54 pares"
print(f"Máxima diferencia en valores p de los 54 pares: {(pares['valor_p_10cat'] - pares['valor_p_6cat']).abs().max():.2e}")

resumen_bh = pd.DataFrame({
    "version": ["10 categorías (90 pruebas)", "6 categorías (54 pruebas)"],
    "significativos_con_vcov_corregida": [int((pares["clasificacion_10cat"] != "sin efecto").sum()), int((pares["clasificacion_6cat"] != "sin efecto").sum())],
    "significativos_sin_corregir_vcov": [int((pares["clasificacion_sin_corregir_10cat"] != "sin efecto").sum()), int((pares["clasificacion_sin_corregir_6cat"] != "sin efecto").sum())],
})
print()
print(resumen_bh.to_string(index=False))
print("\nPares significativos SIN corregir la matriz de varianzas (los mismos que obtiene la compañera en 06.5):")
print(cruzados.loc[cruzados["significativo_bh_sin_corregir"],
                   ["categoria_afectada", "categoria_en_promocion", "L_jk_pct", "valor_p_sin_corregir", "valor_q_sin_corregir"]].round(4).to_string(index=False))
pares.to_csv(os.path.join(ruta_tablas, "tabla_31_fase2c_comparacion_con_fase2_de_10.csv"), index=False)

Máxima diferencia en coeficientes frente a la versión de 10 categorías: 2.03e-10
Máxima diferencia en valores p de los 54 pares: 1.75e-06

                   version  significativos_con_vcov_corregida  significativos_sin_corregir_vcov
10 categorías (90 pruebas)                                  0                                 3
 6 categorías (54 pruebas)                                  0                                 3

Pares significativos SIN corregir la matriz de varianzas (los mismos que obtiene la compañera en 06.5):
categoria_afectada categoria_en_promocion  L_jk_pct  valor_p_sin_corregir  valor_q_sin_corregir
  LIQUOR,WINE,BEER                  MEATS    1.9997                0.0003                0.0144
  LIQUOR,WINE,BEER                POULTRY  -10.8461                0.0019                0.0341
  LIQUOR,WINE,BEER              GROCERY I   25.8312                0.0013                0.0341


**Cómo leer esta salida.**
- **Máxima diferencia en coeficientes: 2.03e-10.** Al usar solo 6 categorías, los coeficientes son **idénticos** a los de la versión de 10 categorías (cada categoría se estima por separado). Era lo esperado.
- **Máxima diferencia en valores p: 1.75e-06.** Lo mismo: solo hay diferencias diminutas por redondeo del computador.
- **La tabla de significativos:** con la matriz corregida, **0** significativos tanto con 90 como con 54 pruebas; sin corregir, **3** en los dos casos. **Esto es importante:** el hecho de pasar de 90 a 54 pruebas (menos exigente) **no cambia** la conclusión. Lo que de verdad hace la diferencia entre "0" y "3" es **corregir o no la matriz de varianzas**.
- **Los 3 pares sin corregir** son los mismos que obtiene la compañera en su notebook 06.5, y los valores q coinciden con los suyos (por ejemplo, licores–carnes q = 0,0144).

**Qué corregí o agregué respecto de las dos versiones de origen:**
1. **De la compañera:** solo 6 categorías, diagnóstico de nulos, igualdad exacta de 532 días.
2. **Del 06 explicado:** corrección de la matriz de varianzas con la versión sin corregir como sensibilidad, efecto neto, verificación con statsmodels, controles de estabilidad y explicaciones.
3. **Avisos:** silenciado solo el de `sqrt` (el código ya lo corrige).
4. **No se pisan archivos:** salidas nuevas `tabla_28` a `tabla_31`.

## 11. Verificación con IA (hacer en una conversación NUEVA)

`INSTRUCCIONES.md` (sección 10.2) pide revisar el código con una IA **en una conversación distinta de la que lo generó**, para que no defienda su propio trabajo. Pasos:
1. Abre un chat nuevo.
2. Adjunta siempre **`metodologia.tex`** y este notebook (`06.COMPLETO_fase2_efectos_cruzados.ipynb`).
3. Pega un prompt a la vez y guarda la respuesta.
4. Anota en la tabla del final qué encontró y qué se corrigió.

Archivos que conviene adjuntar según el prompt: **prompts 1, 2 y 3** → metodología + notebook; **prompt 4** → metodología + `tabla_28_fase2c_coeficientes.csv` y `tabla_29_fase2c_efectos_cruzados.csv`; **prompt 5** → el borrador de resultados + las mismas tablas.

**Prompt 1 · Revisión del código contra la metodología**
```
Te adjunto la metodología de mi tesis (metodologia.tex) y un notebook de Python que estima
el modelo de la Fase 2 (efectos de la promoción de otras categorías) para las 6 categorías
objetivo sin programa semanal. Actúa como revisor metodológico exigente. Revisa línea por línea si el código
implementa exactamente la ecuación de la Fase 2 de la metodología: variable dependiente,
cada regresor y sus rezagos (día actual, bloque de 1 a 3 días y bloque de 4 a 6 días),
las otras 9 categorías del núcleo sin la propia, la variable de control p__RESTO, efectos
fijos (SKU, día de la semana, mes), exclusiones (ventana del terremoto, días de programa,
nulos) y agrupamiento de los errores estándar por SKU y semana. Para cada diferencia,
indica la línea, qué dice la metodología, qué hace el código y por qué importa. No
reescribas el código completo: solo señala los problemas y propone la corrección mínima.
Si algo está correcto, dilo explícitamente.
```

**Prompt 2 · Revisión de las uniones y del tamaño de la base**
```
Revisa en este código cada unión (merge/join) y cada filtro de la función armar_base.
Para cada uno, dime: (1) qué claves usa; (2) si la relación es uno a uno, muchos a uno o
muchos a muchos; (3) si puede duplicar o perder filas; (4) qué assert debería agregar para
comprobarlo. La base final debe tener una fila por SKU-día. Señala cualquier paso donde el
tipo de la columna date pueda no coincidir entre tablas (por ejemplo, una leída desde CSV
sin parse_dates). Verifica además que ninguna categoría con programa semanal sea objetivo y que cada
categoría objetivo use exactamente 532 días.
```

**Prompt 3 · Búsqueda adversarial de errores**
```
Supón que este código tiene al menos tres errores que cambian los resultados sin producir
ningún mensaje de error. Encuéntralos. Considera en particular: variables que se filtran
del futuro (por ejemplo, rezagos mal construidos), columnas usadas con un nombre
equivocado, filas excluidas de más o de menos, categorías que deberían eliminarse del
modelo y no se eliminaron (la propia categoría objetivo entre las "otras"), la corrección
de la matriz de varianzas con valores propios negativos, la corrección de Benjamini-
Hochberg sobre los 54 efectos cruzados y errores estándar mal agrupados. Para cada
sospecha, explica cómo comprobarla con una línea de código o un conteo, sin modificar
los datos originales.
```

**Prompt 4 · Interpretación de los resultados**
```
Te adjunto las tablas de resultados de mi modelo PPML de la Fase 2 (coeficientes y efectos
cruzados) y la metodología. Para cada coeficiente relevante, explica en una frase qué
significa en porcentaje de ventas, si es estadísticamente significativo con la corrección
indicada en la metodología (Benjamini-Hochberg, FDR 5 %), y si su signo y magnitud son
plausibles para un supermercado. Señala cualquier resultado que parezca demasiado grande,
demasiado preciso o contraintuitivo (por ejemplo, un efecto cruzado mayor que 20 %), y qué
revisarías antes de reportarlo. No afirmes causalidad más allá de lo que permite el diseño
descrito en las limitaciones.
```

**Prompt 5 · Coherencia entre el texto y los resultados**
```
Te adjunto el borrador de mi sección de resultados de la Fase 2 y las tablas exportadas por
el código. Verifica que cada cifra del texto coincida exactamente con las tablas, que cada
afirmación esté respaldada por un resultado y que no se presenten como efectos nulos los
efectos de los programas semanales fijos, que según la metodología no se pueden estimar.
Haz una lista de las discrepancias.
```

**Registro de la verificación** (revisión hecha en una conversación nueva el 22‑10‑2026; cada hallazgo fue comprobado con los datos):

| Prompt | Qué encontró la IA | ¿Se comprobó? | ¿Se corrigió? | Cómo |
|---|---|---|---|---|
| 1 | La metodología dice 544 días; el código usa 532 | Sí: son 12 días sin historia de rezagos | **Pendiente** (texto de la metodología) | Actualizar el número en el documento |
| 1 | Las categorías con programa no se estiman como objetivo (54 pruebas en vez de 90) | Sí | Declarado en el título de este notebook | Pendiente D‑14 |
| 1 | La corrección de Cameron‑Gelbach‑Miller no está descrita en la metodología | Sí (solo aparece en las referencias) | **Pendiente** | Agregarla en la sección de inferencia |
| 1 | El control "cruzados mucho menores que el propio" es falso con el máximo | Sí (0,94 > 0,46) | **Corregido** | Se compara la mediana y se explica que las unidades no son comparables |
| 1 | El control de coeficientes mayores que 5 solo miraba los 54 de hoy | Sí | **Corregido** | Ahora cubre también los rezagos (`max_abs_coef_promo_todos`) |
| 2 | Faltan controles de cobertura de fechas, tipo de dato, 46 días de terremoto y `'nan'` como texto | Sí | **Corregido** | Agregados en la lectura de insumos |
| 2 | La fase 1 podría tener más de una fila por categoría | — | **Corregido** | `assert fase1.index.is_unique` |
| 3‑A | Datos faltantes estructurales en `promo_clase_otros` (LIQUOR) | Sí: 12,6 % de las filas y 16 de 60 SKUs | **Documentado** | Sección 9; alternativa pendiente con el profesor |
| 3‑B | Definición de `semana` y `mes` | No era problema: semanas con año (2016‑S01…) y meses con año; sin `'nan'` | — | Se imprime el conteo |
| 3‑C | Matriz de varianzas inválida; tercera forma de agrupar | Sí | **Agregada** | Sección 8: con agrupación solo por semana, 0 de 54 significativos |
| 3‑D | Calendario perdido por colinealidad | Se comprobó: ninguna variable se perdió | — | — |
| 3‑E | +10 puntos podría ser extrapolación | Sí: la desviación estándar de abarrotes es 0,03 | **Agregado** | Columna `L_jk_1desv_pct` |
| 3‑F | `p__RESTO` idéntico entre objetivos | Sí (diferencia 0) | — | — |
| 3‑G | Rezago `_l1_3` incluye solo días anteriores | Sí: la diferencia máxima es 7e‑08 (redondeo de 32 bits); el chequeo propuesto tenía una tolerancia demasiado estricta y no tenía en cuenta la ventana del terremoto | — | — |
| 3 | La etiqueta `valor_p_neto_sin_corregir` estaba mal | Sí: se calculaba con la matriz corregida | **Corregido** | Renombrada a `valor_p_neto` |
| 3 | El texto mencionaba un aviso de `separation` que no aparecía | Sí | **Corregido** | Texto ajustado |
| 3 | Cifras del texto no calculadas en código | Sí | **Corregido** | Celda de resumen numérico (sección 5) |
| 3 | statsmodels valida coeficientes pero no errores estándar | Sí | **Aclarado** | Nota en la verificación y sección 8 |

Recuerda que la IA nueva también puede equivocarse: cada hallazgo hay que comprobarlo con un conteo o una línea de código antes de cambiar nada (aquí el chequeo del rezago resultó ser un falso positivo).
